In [ ]:
!pip install torch_geometric
!pip install torch-geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster \
  -f https://data.pyg.org/whl/torch-2.10.0+cu128.html

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import xgboost as xgb
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from math import log
import warnings
warnings.filterwarnings('ignore')

# 1. CHARGEMENT ET RÉINDEXATION

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:


path = "/content/drive/MyDrive/MLNS/"
node_info  = pd.read_csv(path + 'node_information.csv', header=None, index_col=0)
train_data = pd.read_csv(path + 'train.txt', sep=' ', header=None, names=['source', 'target', 'label'])
test_data  = pd.read_csv(path + 'test.txt',  sep=' ', header=None, names=['source', 'target'])
print(f"Nœuds: {len(node_info)}, Train: {len(train_data)}, Test: {len(test_data)}")

mapping = {oid: nid for nid, oid in enumerate(node_info.index)}
for df in [train_data, test_data]:
    df['source'] = df['source'].map(mapping)
    df['target'] = df['target'].map(mapping)
train_data = train_data.dropna().astype(int)
test_data  = test_data.dropna().astype(int)
N_NODES    = len(node_info)

# 2. FEATURES TEXTUELLES — SVD 128

In [ ]:
raw_matrix = node_info.values.astype(np.float32)
print("\nSVD 128 composantes...")
svd     = TruncatedSVD(n_components=128, random_state=42)
svd_mat = svd.fit_transform(raw_matrix).astype(np.float32)
print(f"Variance expliquée: {svd.explained_variance_ratio_.sum():.4f}")

# 3. EXTRACTION DES FEATURES

In [ ]:
def build_community_kcore(graph):
    try:
        parts = nx.community.louvain_communities(graph, seed=42)
        partition  = {}
        comm_sizes = {}
        for cid, comm in enumerate(parts):
            for node in comm:
                partition[node]  = cid
                comm_sizes[node] = len(comm)
        print(f"    ✓ Louvain : {len(parts)} communautés")
    except Exception:
        partition, comm_sizes = {}, {}
        print("    ✗ Louvain indisponible")

    try:
        cores = nx.core_number(graph)
    except Exception:
        cores = {}

    return partition, comm_sizes, cores

In [ ]:
def compute_neigh_embeddings(graph, svd_mat, N_NODES):
    """GNN 1-hop : moyenne des embeddings SVD des voisins."""
    neigh = np.zeros_like(svd_mat)
    for u in range(N_NODES):
        nbrs = list(graph.neighbors(u))
        if nbrs:
            neigh[u] = svd_mat[nbrs].mean(axis=0)
    return neigh

In [ ]:
def extract_features(df, graph, neigh_emb, partition, comm_sizes, cores):
    features  = []
    degrees   = dict(graph.degree())
    pagerank  = nx.pagerank(graph, alpha=0.85, max_iter=100)
    clustering = nx.clustering(graph)

    for _, row in df.iterrows():
        u, v = int(row['source']), int(row['target'])

        deg_u = degrees.get(u, 0)
        deg_v = degrees.get(v, 0)

        # ── Topologie ─────────────────────────────────────────────────
        common_neigh = jaccard = adamic_adar = ra_index = 0.0
        salton = sorensen = 0.0

        if graph.has_node(u) and graph.has_node(v):
            neighbors    = list(nx.common_neighbors(graph, u, v))
            common_neigh = len(neighbors)
            set_u = set(graph[u])
            set_v = set(graph[v])
            union_size   = len(set_u | set_v)

            if union_size > 0:
                jaccard = common_neigh / union_size
            salton   = common_neigh / (np.sqrt(deg_u * deg_v) + 1e-8)
            sorensen = 2 * common_neigh / (deg_u + deg_v + 1e-8)

            for w in neighbors:
                d_w = degrees.get(w, 0)
                if d_w > 1:
                    adamic_adar += 1.0 / log(d_w)
                    ra_index    += 1.0 / d_w

        pr_prod = pagerank.get(u, 0.0) * pagerank.get(v, 0.0)
        cc_u    = clustering.get(u, 0.0)
        cc_v    = clustering.get(v, 0.0)

        try:
            sp = nx.shortest_path_length(graph, u, v)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            sp = 10

        # ── SVD texte ─────────────────────────────────────────────────
        svd_u = svd_mat[u]
        svd_v = svd_mat[v]
        svd_cos       = np.dot(svd_u, svd_v) / (np.linalg.norm(svd_u) * np.linalg.norm(svd_v) + 1e-8)
        svd_euclidean = np.linalg.norm(svd_u - svd_v)
        hadamard_svd  = svd_u * svd_v
        diff_abs_svd  = np.abs(svd_u - svd_v)
        svd_sum       = svd_u + svd_v

        # ── Texte brut ────────────────────────────────────────────────
        raw_u  = raw_matrix[u]
        raw_v  = raw_matrix[v]
        raw_dot = float(np.dot(raw_u, raw_v))
        raw_cos = raw_dot / (np.linalg.norm(raw_u) * np.linalg.norm(raw_v) + 1e-8)

        # ── GNN scalaire ──────────────────────────────────────────────
        n_u = neigh_emb[u]
        n_v = neigh_emb[v]
        neigh_cos = float(np.dot(n_u, n_v) / (np.linalg.norm(n_u) * np.linalg.norm(n_v) + 1e-8))

        # ── Community ─────────────────────────────────────────────────
        comm_u          = partition.get(u, -1)
        comm_v          = partition.get(v, -1)
        same_community  = float(comm_u == comm_v and comm_u != -1)
        cs_u            = float(comm_sizes.get(u, 1))
        cs_v            = float(comm_sizes.get(v, 1))
        cs_ratio        = min(cs_u, cs_v) / (max(cs_u, cs_v) + 1e-8)

        # ── K-core ────────────────────────────────────────────────────
        kc_u   = float(cores.get(u, 0))
        kc_v   = float(cores.get(v, 0))
        kc_min = min(kc_u, kc_v)
        kc_diff = abs(kc_u - kc_v)

        # ── Assemblage ────────────────────────────────────────────────
        base = [
            common_neigh, jaccard, adamic_adar, ra_index,
            salton, sorensen,
            deg_u, deg_v, deg_u * deg_v,
            sp,
            pr_prod,
            cc_u, cc_v, cc_u * cc_v,
            svd_cos, svd_euclidean,
            raw_dot, raw_cos,
            neigh_cos,
            same_community, cs_u, cs_v, cs_ratio,
            kc_u, kc_v, kc_min, kc_diff,
        ]

        features.append(np.concatenate([base, hadamard_svd, diff_abs_svd, svd_sum]))

    return np.array(features, dtype=np.float32)


# 4. STRATIFIED K-FOLD (10 folds)

In [ ]:
print("\n" + "="*55)
print("K-FOLD STRATIFIÉ — 10 FOLDS")
print("="*55)

N_SPLITS   = 10
skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
y_full     = train_data['label'].values
test_preds = np.zeros(len(test_data))
oof_preds  = np.zeros(len(train_data))

for fold, (train_idx, val_idx) in enumerate(skf.split(train_data[['source','target']].values, y_full)):
    print(f"\n{'='*55}")
    print(f"FOLD {fold+1}/{N_SPLITS}")
    print(f"{'='*55}")

    fold_train = train_data.iloc[train_idx]
    fold_val   = train_data.iloc[val_idx]

    G = nx.Graph()
    G.add_nodes_from(range(N_NODES))
    G.add_edges_from(fold_train[fold_train['label'] == 1][['source','target']].values)
    print(f"Graphe: {G.number_of_nodes()} nœuds, {G.number_of_edges()} arêtes")

    print("  Precompute community + kcore...")
    partition, comm_sizes, cores = build_community_kcore(G)

    print("  Precompute neighborhood embeddings...")
    neigh_emb = compute_neigh_embeddings(G, svd_mat, N_NODES)

    print("  Extraction features train...")
    X_tr  = extract_features(fold_train, G, neigh_emb, partition, comm_sizes, cores)
    y_tr  = fold_train['label'].values

    print("  Extraction features val...")
    X_val = extract_features(fold_val, G, neigh_emb, partition, comm_sizes, cores)
    y_val = fold_val['label'].values
    print(f"  Shape: {X_tr.shape}")

    clf = xgb.XGBClassifier(
    n_estimators          = 10000,   # 2500 → 10000 (plafond, early stopping coupera)
    max_depth             = 5,
    learning_rate         = 0.005,   # 0.01 → 0.005 (2× plus lent, 2× plus fin)
    subsample             = 0.8,
    colsample_bytree      = 0.25,
    gamma                 = 23,
    min_child_weight      = 1,
    reg_lambda            = 1.0,
    eval_metric           = 'auc',
    early_stopping_rounds = 300,     # 100 → 300 (plus de patience, lr plus petit)
    random_state          = 42,
    n_jobs                = -1,
    )
    clf.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=300)

    oof_preds[val_idx] = clf.predict_proba(X_val)[:, 1]
    print(f"  Validation AUC: {roc_auc_score(y_val, oof_preds[val_idx]):.4f}")

    X_test      = extract_features(test_data, G, neigh_emb, partition, comm_sizes, cores)
    test_preds += clf.predict_proba(X_test)[:, 1] / N_SPLITS
    print(f"✓ FOLD {fold+1} terminé")

# 5. SOUMISSION

In [ ]:
print("\n" + "="*55)
submission = pd.DataFrame({'ID': range(len(test_preds)), 'Predicted': test_preds})
submission.to_csv('submission_v5.csv', index=False)
print("✓ 'submission_v5.csv' généré")
print(f"OOF Global AUC : {roc_auc_score(y_full, oof_preds):.4f}")
print(f"Test — min: {test_preds.min():.4f}, max: {test_preds.max():.4f}, mean: {test_preds.mean():.4f}")